# CAPSTONE PROJECT: TOPIC 06 - UPI TRANSACTION FRAUD SIGNALS
## Enterprise Medallion Data Engineering Pipeline on Databricks & Snowflake
**Student Name:** Siddharth Sonkar  
**Roll Number:** 23051628  
**Domain:** FinTech & Financial Payments Analytics  
**Target Platform:** Databricks (PySpark / Delta Lake) + Snowflake (SQL Warehouse)


### 1. Catalog, Schema & Volume Setup
Define volume paths for Bronze raw data ingestion.

In [ ]:
# Define Volume Path & Seeds
MY_ID = "23051628_SiddharthSonkar"
VOL = f"/Volumes/workspace/capstone_{MY_ID}/raw"
print(f"Target Storage Volume: {VOL}")

### 2. Bronze Layer Ingestion
Ingest raw CSV datasets without altering raw columns, adding audit metadata (`_source_file`, `_ingested_at`, `_row_hash`).

In [ ]:
import pyspark.sql.functions as F

# Bronze Accounts Ingestion
bronze_accounts = (spark.read.option("header", True).csv(f"{VOL}/accounts.csv")
    .withColumn("_source_file", F.input_file_name())
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_row_hash", F.sha2(F.concat_ws("||", "account_id", "bank", "home_city", "primary_device", "secondary_device"), 256)))
bronze_accounts.write.mode("overwrite").saveAsTable("bronze_accounts")
print(f"Bronze Accounts: {bronze_accounts.count():,} rows")

# Bronze Transactions Ingestion
bronze_txns = (spark.read.option("header", True).csv(f"{VOL}/txns.csv")
    .withColumn("_source_file", F.input_file_name())
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_row_hash", F.sha2(F.concat_ws("||", "txn_id", "account_id", "merchant_id", "merchant_category", "txn_ts", "device_id", "status", "amount"), 256)))
bronze_txns.write.mode("overwrite").saveAsTable("bronze_txns")
print(f"Bronze Transactions: {bronze_txns.count():,} rows")

### 3. Silver Layer Cleaning & Rejects Framework
Perform deduplication on `txn_id`, strong type casting with `try_cast`, and route defective records to `silver_rejects`.

In [ ]:
# Deduplicate on txn_id
deduped_txns = bronze_txns.dropDuplicates(["txn_id"])

# Type Parsing
parsed = deduped_txns.withColumn("amount_cast", F.expr("try_cast(amount as decimal(18,2))")) \
                     .withColumn("txn_ts_cast", F.expr("try_cast(txn_ts as timestamp)"))

# Rejects Isolation
rej_na = parsed.filter(F.col("amount_cast").isNull()).select("txn_id", "account_id", "merchant_id", "txn_ts", F.lit("unparseable_amount").alias("reject_reason"))
valid_amt = parsed.filter(F.col("amount_cast").isNotNull())

rej_acc = valid_amt.filter(F.col("account_id") == "ACC99999999").select("txn_id", "account_id", "merchant_id", "txn_ts", F.lit("unknown_account").alias("reject_reason"))
valid_acc = valid_amt.filter(F.col("account_id") != "ACC99999999")

rej_ts = valid_acc.filter((F.col("txn_ts_cast") < F.to_timestamp(F.lit("2025-05-01 00:00:00"))) | (F.col("txn_ts_cast") > F.to_timestamp(F.lit("2025-06-14 23:59:59")))) \
    .select("txn_id", "account_id", "merchant_id", "txn_ts", F.lit("timestamp_out_of_range").alias("reject_reason"))

silver_rejects = rej_na.unionByName(rej_acc).unionByName(rej_ts).withColumn("_rejected_at", F.current_timestamp())
silver_rejects.write.mode("overwrite").saveAsTable("silver_rejects")

# Clean Silver Transactions
silver_txns = valid_acc.filter((F.col("txn_ts_cast") >= F.to_timestamp(F.lit("2025-05-01 00:00:00"))) & (F.col("txn_ts_cast") <= F.to_timestamp(F.lit("2025-06-14 23:59:59")))) \
    .withColumn("amount", F.col("amount_cast")).withColumn("txn_ts", F.col("txn_ts_cast")).withColumn("account_day", F.to_date(F.col("txn_ts")))
silver_txns.write.mode("overwrite").saveAsTable("silver_txns")
print(f"Silver Rejects: {silver_rejects.count():,} rows | Silver Clean Txns: {silver_txns.count():,} rows")

### 4. Gold Layer Transformations & Window Baselining
Compute `GOLD_ACCOUNT_DAY` using `ROWS BETWEEN 30 PRECEDING AND 1 PRECEDING` to avoid baseline self-contamination.

In [ ]:
from pyspark.sql.window import Window

# Daily aggregation
daily = silver_txns.groupBy("account_id", "account_day").agg(
    F.count("txn_id").alias("txns"),
    F.max("amount").alias("max_amount"),
    F.countDistinct("device_id").alias("distinct_devices")
)

# Non-contaminating 30-day trailing window frame
w30 = Window.partitionBy("account_id").orderBy("account_day").rowsBetween(-30, -1)

gold_account_day = daily.withColumn("avg_amount_30d_trailing", F.avg("max_amount").over(w30)) \
    .withColumn("prior_txn_count", F.count("max_amount").over(w30)) \
    .withColumn("is_amount_spike", F.when((F.col("prior_txn_count") >= 5) & (F.col("max_amount") >= 5 * F.col("avg_amount_30d_trailing")), True).otherwise(False)) \
    .withColumn("is_device_burst", F.when(F.col("distinct_devices") >= 3, True).otherwise(False))

gold_account_day.write.mode("overwrite").saveAsTable("gold_account_day")
print(f"GOLD_ACCOUNT_DAY written: {gold_account_day.count():,} rows")

### 5. Snowflake Query Suite & Dynamic Data Masking
Analytical queries for business inquiry answers.

In [ ]:
%sql
-- Q1: Flagged Amount Spikes (5x Baseline)
SELECT account_id, account_day, max_amount, avg_amount_30d_trailing AS baseline_30d, prior_txn_count
FROM gold_account_day
WHERE is_amount_spike = true
ORDER BY max_amount / avg_amount_30d_trailing DESC;